# Modelagem dimensional e análise em SQL

Modelo estrela construído sobre exportações brutas de ERP, com todo o tratamento, a modelagem e a análise escritos em SQL.

Roda em **DuckDB** dentro do Colab. Sem instalação local, sem servidor e sem custo.

> ### ⚠️ Dados fictícios
> A Casa Verde Distribuidora é uma empresa inventada e os dados são gerados por script. A modelagem e as consultas são reais.

**O caminho:** três CSVs sujos → camada de staging → dimensões e fato → consultas de negócio → painel no Looker Studio.

## Preparação

In [ ]:
!pip install duckdb --quiet
!git clone -q https://github.com/SEU_USUARIO/casa-verde-margem.git
%cd casa-verde-margem
!python src/gerar_dados.py

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

con = duckdb.connect()

# Os arquivos entram como estao, sem tratamento previo em Python.
# all_varchar evita que o leitor adivinhe tipos numa base inconsistente.
con.execute("""
CREATE VIEW raw_vendas AS
    SELECT * FROM read_csv('data/raw/vendas.csv',   delim=';', header=true, all_varchar=true);
CREATE VIEW raw_produtos AS
    SELECT * FROM read_csv('data/raw/produtos.csv', delim=';', header=true, all_varchar=true);
CREATE VIEW raw_clientes AS
    SELECT * FROM read_csv('data/raw/clientes.csv', delim=',', header=true, all_varchar=true);
""")

con.sql("SELECT * FROM raw_vendas LIMIT 5").df()

## Staging

Macros de conversão e views que padronizam as três origens.

In [ ]:
con.execute("""
CREATE OR REPLACE MACRO to_num(v) AS TRY_CAST(
    CASE
        WHEN contains(trim(replace(replace(CAST(v AS VARCHAR), 'R$', ''), ' ', '')), '.')
         AND contains(trim(replace(replace(CAST(v AS VARCHAR), 'R$', ''), ' ', '')), ',')
        THEN replace(replace(trim(replace(replace(CAST(v AS VARCHAR), 'R$', ''), ' ', '')), '.', ''), ',', '.')
        ELSE replace(trim(replace(replace(CAST(v AS VARCHAR), 'R$', ''), ' ', '')), ',', '.')
    END AS DOUBLE);

-- O formato e escolhido pelo padrao do texto, e nao por tentativa em
-- cadeia. O strptime aceita '%Y' com dois digitos, entao '05-03-25'
-- seria lido como ano 5 se a data curta fosse testada depois da ISO.
CREATE OR REPLACE MACRO to_date_br(v) AS TRY_CAST(
    CASE
        WHEN regexp_matches(trim(CAST(v AS VARCHAR)), '^\d{2}/\d{2}/\d{4}$')
            THEN try_strptime(trim(CAST(v AS VARCHAR)), '%d/%m/%Y')
        WHEN regexp_matches(trim(CAST(v AS VARCHAR)), '^\d{4}-\d{2}-\d{2}$')
            THEN try_strptime(trim(CAST(v AS VARCHAR)), '%Y-%m-%d')
        WHEN regexp_matches(trim(CAST(v AS VARCHAR)), '^\d{2}-\d{2}-\d{2}$')
            THEN try_strptime(trim(CAST(v AS VARCHAR)), '%d-%m-%y')
    END AS DATE);

CREATE OR REPLACE MACRO norm_txt(v) AS
    nullif(regexp_replace(trim(CAST(v AS VARCHAR)), '\s+', ' ', 'g'), '');

-- ---------------------------------------------------------------------
""")

con.sql("SELECT to_num('R$ 1.234,56') AS valor, to_date_br('05-03-25') AS data_curta, to_date_br('03/05/2025') AS data_br").df()

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW stg_produtos AS
SELECT
    codigo_produto,
    norm_txt(descricao)          AS descricao,
    norm_txt(categoria)          AS categoria,
    norm_txt(unidade)            AS unidade,
    to_num(custo_unitario)       AS custo_unitario,
    to_num(preco_tabela)         AS preco_tabela,
    upper(trim(ativo)) = 'S'     AS ativo
FROM raw_produtos;

CREATE OR REPLACE VIEW stg_clientes AS
SELECT
    codigo_cliente,
    norm_txt(razao_social)                        AS razao_social,
    nullif(regexp_replace(cnpj, '\D', '', 'g'), '') AS cnpj,
    norm_txt(segmento)                            AS segmento,
    norm_txt(regiao)                              AS regiao,
    trim(vendedor)                                AS cod_vendedor,
    to_date_br(data_cadastro)                     AS data_cadastro,
    to_num(limite_credito)                        AS limite_credito
FROM raw_clientes;

-- Deduplicacao pela chave de negocio do ERP e exclusao de cancelados.
-- Quantidade negativa e devolucao lancada sem sinalizacao propria.
CREATE OR REPLACE VIEW stg_vendas AS
WITH base AS (
    SELECT
        CAST(numero_pedido AS INTEGER)   AS numero_pedido,
        CAST(sequencia_item AS INTEGER)  AS sequencia_item,
        to_date_br(data_pedido)          AS data_pedido,
        codigo_cliente,
        codigo_produto,
        CAST(quantidade AS INTEGER)      AS quantidade,
        to_num(preco_unitario)           AS preco_unitario,
        to_num(desconto_pct)             AS desconto_pct,
        to_num(frete_cobrado)            AS frete_cobrado,
        to_num(custo_frete_real)         AS custo_frete_real,
        trim(vendedor)                   AS cod_vendedor,
        upper(trim(status))              AS status,
        -- A chave pedido mais item e unica no ERP, entao repeticao e
        -- falha de exportacao. O criterio de desempate e explicito para
        -- o resultado nao depender da ordem em que o arquivo foi lido.
        row_number() OVER (
            PARTITION BY numero_pedido, sequencia_item
            ORDER BY CASE WHEN CAST(quantidade AS INTEGER) < 0 THEN 1 ELSE 0 END,
                     codigo_cliente
        ) AS ocorrencia
    FROM raw_vendas
)
SELECT
    * EXCLUDE (ocorrencia),
    CASE WHEN quantidade < 0 THEN 'DEVOLUCAO' ELSE 'VENDA' END AS tipo_operacao
FROM base
WHERE ocorrencia = 1
  AND status <> 'CANCELADO'
  AND data_pedido IS NOT NULL;
""")

con.sql("SELECT * FROM stg_vendas LIMIT 5").df()

In [ ]:
con.sql("""
SELECT
    (SELECT count(*) FROM raw_vendas)  AS linhas_brutas,
    (SELECT count(*) FROM stg_vendas)  AS linhas_tratadas,
    (SELECT count(*) FROM raw_vendas) - (SELECT count(*) FROM stg_vendas) AS descartadas,
    (SELECT count(*) FROM stg_vendas WHERE tipo_operacao = 'DEVOLUCAO')   AS devolucoes,
    (SELECT count(*) FROM stg_produtos WHERE custo_unitario IS NULL)      AS produtos_sem_custo
""").df()

## Dimensões

Chaves substitutas próprias, independentes do código do sistema de origem. A dimensão cliente recebe um membro para pedidos órfãos, o que evita perder receita no cruzamento.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dim_produto AS
SELECT
    row_number() OVER (ORDER BY codigo_produto) AS sk_produto,
    codigo_produto,
    descricao,
    categoria,
    unidade,
    custo_unitario,
    preco_tabela,
    custo_unitario IS NOT NULL AS custo_confiavel,
    ativo
FROM stg_produtos;

CREATE OR REPLACE TABLE dim_cliente AS
SELECT
    row_number() OVER (ORDER BY codigo_cliente) AS sk_cliente,
    codigo_cliente,
    razao_social,
    cnpj,
    segmento,
    regiao,
    data_cadastro
FROM stg_clientes
UNION ALL
SELECT 0, 'NAO_IDENTIFICADO', 'Cliente nao identificado', NULL, NULL, NULL, NULL;

CREATE OR REPLACE TABLE dim_vendedor AS
SELECT
    row_number() OVER (ORDER BY cod_vendedor) AS sk_vendedor,
    cod_vendedor,
    CASE cod_vendedor
        WHEN 'V01' THEN 'Rogerio Antunes'
        WHEN 'V02' THEN 'Simone Klein'
        WHEN 'V03' THEN 'Tarcisio Bueno'
        ELSE 'Nao informado'
    END AS nome_vendedor,
    CASE cod_vendedor
        WHEN 'V01' THEN 0.030
        WHEN 'V02' THEN 0.035
        WHEN 'V03' THEN 0.028
        ELSE 0.032
    END AS taxa_comissao
FROM (SELECT DISTINCT cod_vendedor FROM stg_vendas WHERE cod_vendedor IS NOT NULL);

CREATE OR REPLACE TABLE dim_tempo AS
SELECT
    CAST(strftime(d, '%Y%m%d') AS INTEGER) AS sk_tempo,
    d                                      AS data,
    year(d)                                AS ano,
    month(d)                               AS mes,
    strftime(d, '%Y-%m')                   AS ano_mes,
    quarter(d)                             AS trimestre,
    dayofweek(d)                           AS dia_semana,
    strftime(d, '%A')                      AS nome_dia,
    dayofweek(d) IN (0, 6)                 AS fim_de_semana
FROM (
    SELECT (lim.inicio + to_days(CAST(g.n AS INTEGER)))::DATE AS d
    FROM (SELECT min(data_pedido) AS inicio, max(data_pedido) AS fim FROM stg_vendas) AS lim,
         generate_series(
             0,
             (SELECT datediff('day', min(data_pedido), max(data_pedido)) FROM stg_vendas)
         ) AS g(n)
);
""")

con.sql("SELECT 'produto' t, count(*) FROM dim_produto UNION ALL SELECT 'cliente', count(*) FROM dim_cliente UNION ALL SELECT 'vendedor', count(*) FROM dim_vendedor UNION ALL SELECT 'tempo', count(*) FROM dim_tempo").df()

## Fato

Grão de uma linha por item de pedido. As medidas seguem a conta que o ERP não faz, descontando a entrega real e a comissão.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE fct_vendas AS
SELECT
    v.numero_pedido,
    v.sequencia_item,
    t.sk_tempo,
    coalesce(c.sk_cliente, 0)  AS sk_cliente,
    p.sk_produto,
    ve.sk_vendedor,
    v.tipo_operacao,
    v.quantidade,
    v.preco_unitario,
    v.desconto_pct,

    -- medidas aditivas
    round(v.preco_unitario * v.quantidade, 2)                       AS receita_mercadoria,
    round(v.preco_unitario * v.quantidade + v.frete_cobrado, 2)     AS receita_total,
    round(p.custo_unitario * v.quantidade, 2)                       AS custo_produto,
    round(v.custo_frete_real, 2)                                    AS custo_entrega,
    round(v.preco_unitario * v.quantidade * ve.taxa_comissao, 2)    AS custo_comissao,
    round(v.custo_frete_real - v.frete_cobrado, 2)                  AS frete_subsidiado,

    -- margem bruta e o que o ERP nao mostra
    CASE WHEN p.custo_confiavel
         THEN round(v.preco_unitario * v.quantidade - p.custo_unitario * v.quantidade, 2)
    END AS margem_bruta,
    CASE WHEN p.custo_confiavel
         THEN round(
              v.preco_unitario * v.quantidade + v.frete_cobrado
            - p.custo_unitario * v.quantidade
            - v.custo_frete_real
            - v.preco_unitario * v.quantidade * ve.taxa_comissao, 2)
    END AS margem_contribuicao,

    p.custo_confiavel
FROM stg_vendas v
JOIN dim_produto  p  ON p.codigo_produto = v.codigo_produto
JOIN dim_tempo    t  ON t.data           = v.data_pedido
JOIN dim_vendedor ve ON ve.cod_vendedor  = v.cod_vendedor
LEFT JOIN dim_cliente c ON c.codigo_cliente = v.codigo_cliente;
""")

con.sql("SELECT * FROM fct_vendas LIMIT 5").df()

In [ ]:
# Validacao: nenhuma chave orfa e o total fecha com a origem
con.sql("""
SELECT
    count(*)                                              AS linhas,
    count(*) FILTER (WHERE sk_produto  IS NULL)           AS sem_produto,
    count(*) FILTER (WHERE sk_tempo    IS NULL)           AS sem_tempo,
    count(*) FILTER (WHERE sk_vendedor IS NULL)           AS sem_vendedor,
    count(*) FILTER (WHERE sk_cliente = 0)                AS cliente_nao_identificado,
    count(*) FILTER (WHERE NOT custo_confiavel)           AS sem_custo,
    round(sum(receita_total), 2)                          AS receita,
    round(sum(margem_contribuicao), 2)                    AS margem
FROM fct_vendas
""").df()

### 01. Onde a margem se forma e onde ela se perde

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW an_categoria AS
SELECT
    p.categoria,
    count(DISTINCT f.numero_pedido)                       AS pedidos,
    round(sum(f.receita_total), 2)                        AS receita,
    round(sum(f.custo_produto), 2)                        AS custo_produto,
    round(sum(f.custo_entrega), 2)                        AS custo_entrega,
    round(sum(f.custo_comissao), 2)                       AS custo_comissao,
    round(sum(f.frete_subsidiado), 2)                     AS frete_subsidiado,
    round(sum(f.margem_bruta), 2)                         AS margem_bruta,
    round(sum(f.margem_contribuicao), 2)                  AS margem_contribuicao,
    round(100.0 * sum(f.margem_contribuicao)
               / nullif(sum(f.receita_total), 0), 1)      AS margem_pct,
    round(100.0 * sum(f.receita_total)
               / sum(sum(f.receita_total)) OVER (), 1)    AS participacao_pct,
    CASE
        WHEN sum(f.margem_contribuicao) < 0 THEN 'Prejuizo'
        WHEN 100.0 * sum(f.margem_contribuicao) / nullif(sum(f.receita_total), 0) < 10 THEN 'Critica'
        WHEN 100.0 * sum(f.margem_contribuicao) / nullif(sum(f.receita_total), 0) < 20 THEN 'Atencao'
        ELSE 'Saudavel'
    END                                                   AS situacao
FROM fct_vendas f
JOIN dim_produto p ON p.sk_produto = f.sk_produto
WHERE f.custo_confiavel
GROUP BY p.categoria
ORDER BY margem_contribuicao;
""")

con.sql("SELECT * FROM an_categoria").df()

### 02. Da receita ao que sobra, em degraus

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW an_cascata AS
WITH t AS (
    SELECT
        sum(receita_total)      AS receita,
        sum(custo_produto)      AS mercadoria,
        sum(custo_entrega)      AS entrega,
        sum(custo_comissao)     AS comissao,
        sum(margem_contribuicao) AS margem
    FROM fct_vendas WHERE custo_confiavel
)
SELECT 1 AS ordem, 'Receita'    AS etapa, round(receita, 2)     AS valor FROM t
UNION ALL SELECT 2, 'Mercadoria', round(-mercadoria, 2) FROM t
UNION ALL SELECT 3, 'Entrega',    round(-entrega, 2)    FROM t
UNION ALL SELECT 4, 'Comissao',   round(-comissao, 2)   FROM t
UNION ALL SELECT 5, 'Margem',     round(margem, 2)      FROM t
ORDER BY ordem;
""")

con.sql("SELECT * FROM an_cascata").df()

### 03. Curva ABC de clientes cruzada com rentabilidade

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW an_clientes_abc AS
WITH por_cliente AS (
    SELECT
        c.codigo_cliente,
        c.razao_social,
        c.segmento,
        c.regiao,
        count(DISTINCT f.numero_pedido)        AS pedidos,
        sum(f.receita_total)                   AS receita,
        sum(f.margem_contribuicao)             AS margem,
        sum(f.frete_subsidiado)                AS frete_subsidiado,
        avg(f.desconto_pct)                    AS desconto_medio
    FROM fct_vendas f
    JOIN dim_cliente c ON c.sk_cliente = f.sk_cliente
    WHERE f.custo_confiavel AND c.codigo_cliente <> 'NAO_IDENTIFICADO'
    GROUP BY ALL
),
acumulado AS (
    SELECT
        *,
        sum(receita) OVER (ORDER BY receita DESC
                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
            / sum(receita) OVER ()                       AS receita_acumulada,
        row_number() OVER (ORDER BY receita DESC)        AS posicao
    FROM por_cliente
)
SELECT
    posicao,
    razao_social,
    segmento,
    regiao,
    pedidos,
    round(receita, 2)                          AS receita,
    round(receita / pedidos, 2)                AS ticket_medio,
    round(margem, 2)                           AS margem,
    round(100.0 * margem / nullif(receita, 0), 1) AS margem_pct,
    round(frete_subsidiado, 2)                 AS frete_subsidiado,
    round(desconto_medio, 1)                   AS desconto_medio,
    round(100.0 * receita_acumulada, 1)        AS receita_acumulada_pct,
    CASE WHEN receita_acumulada <= 0.80 THEN 'A'
         WHEN receita_acumulada <= 0.95 THEN 'B'
         ELSE 'C' END                          AS classe_abc
FROM acumulado
ORDER BY posicao;
""")

con.sql("SELECT * FROM an_clientes_abc LIMIT 12").df()

### 04. Os maiores clientes rendem menos que a media da carteira

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW an_clientes_criticos AS
WITH media AS (
    SELECT 100.0 * sum(margem_contribuicao) / sum(receita_total) AS margem_carteira
    FROM fct_vendas WHERE custo_confiavel
)
SELECT
    a.classe_abc,
    a.razao_social,
    a.segmento,
    a.receita,
    a.margem,
    a.margem_pct,
    round(m.margem_carteira, 1)                       AS margem_carteira_pct,
    round(a.margem_pct - m.margem_carteira, 1)        AS diferenca_pp,
    a.desconto_medio,
    round(a.receita * (m.margem_carteira - a.margem_pct) / 100, 2) AS margem_nao_realizada
FROM an_clientes_abc a
CROSS JOIN media m
WHERE a.classe_abc IN ('A', 'B')
  AND a.margem_pct < m.margem_carteira
ORDER BY margem_nao_realizada DESC;
""")

con.sql("SELECT * FROM an_clientes_criticos").df()

### 05. O ponto em que o desconto deixa de compensar

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW an_desconto AS
SELECT
    CASE
        WHEN desconto_pct < 5  THEN '1. ate 5%'
        WHEN desconto_pct < 10 THEN '2. 5 a 10%'
        WHEN desconto_pct < 15 THEN '3. 10 a 15%'
        WHEN desconto_pct < 20 THEN '4. 15 a 20%'
        ELSE '5. acima de 20%'
    END                                                   AS faixa,
    count(*)                                              AS itens,
    round(sum(receita_total), 2)                          AS receita,
    round(100.0 * sum(receita_total)
               / sum(sum(receita_total)) OVER (), 1)      AS participacao_pct,
    round(sum(margem_contribuicao), 2)                    AS margem,
    round(100.0 * sum(margem_contribuicao)
               / nullif(sum(receita_total), 0), 1)        AS margem_pct
FROM fct_vendas
WHERE custo_confiavel
GROUP BY faixa
ORDER BY faixa;
""")

con.sql("SELECT * FROM an_desconto").df()

### 06. Quanto a politica de frete gratis custa

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW an_frete AS
SELECT
    p.categoria,
    round(sum(f.receita_total), 2)                        AS receita,
    round(sum(f.custo_entrega), 2)                        AS custo_entrega,
    round(sum(f.custo_entrega - f.frete_subsidiado), 2)   AS frete_cobrado,
    round(sum(f.frete_subsidiado), 2)                     AS frete_absorvido,
    round(100.0 * sum(f.frete_subsidiado)
               / nullif(sum(f.receita_total), 0), 1)      AS absorvido_sobre_receita_pct,
    round(sum(f.margem_contribuicao), 2)                  AS margem,
    round(sum(f.margem_contribuicao) + sum(f.frete_subsidiado), 2) AS margem_sem_subsidio
FROM fct_vendas f
JOIN dim_produto p ON p.sk_produto = f.sk_produto
WHERE f.custo_confiavel
GROUP BY p.categoria
ORDER BY frete_absorvido DESC;
""")

con.sql("SELECT * FROM an_frete").df()

### 07. Evolucao mensal com variacao sobre o mes anterior

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW an_mensal AS
WITH mes AS (
    SELECT
        t.ano_mes,
        count(DISTINCT f.numero_pedido)  AS pedidos,
        sum(f.receita_total)             AS receita,
        sum(f.margem_contribuicao)       AS margem,
        sum(f.frete_subsidiado)          AS frete_absorvido
    FROM fct_vendas f
    JOIN dim_tempo t ON t.sk_tempo = f.sk_tempo
    WHERE f.custo_confiavel
    GROUP BY t.ano_mes
)
SELECT
    ano_mes,
    pedidos,
    round(receita, 2)                                        AS receita,
    round(receita / pedidos, 2)                              AS ticket_medio,
    round(margem, 2)                                         AS margem,
    round(100.0 * margem / nullif(receita, 0), 1)            AS margem_pct,
    round(frete_absorvido, 2)                                AS frete_absorvido,
    round(100.0 * (receita - lag(receita) OVER (ORDER BY ano_mes))
               / nullif(lag(receita) OVER (ORDER BY ano_mes), 0), 1) AS var_receita_pct,
    round(100.0 * (margem - lag(margem) OVER (ORDER BY ano_mes))
               / nullif(abs(lag(margem) OVER (ORDER BY ano_mes)), 0), 1) AS var_margem_pct,
    round(avg(100.0 * margem / nullif(receita, 0))
              OVER (ORDER BY ano_mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1)
                                                             AS margem_pct_media_3m
FROM mes
ORDER BY ano_mes;
""")

con.sql("SELECT * FROM an_mensal").df()

### 08. Produtos que vendem e dao prejuizo

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW an_produtos_prejuizo AS
SELECT
    p.codigo_produto,
    p.descricao,
    p.categoria,
    sum(f.quantidade)                                     AS quantidade,
    round(sum(f.receita_total), 2)                        AS receita,
    round(sum(f.margem_bruta), 2)                         AS margem_bruta,
    round(sum(f.margem_contribuicao), 2)                  AS margem_contribuicao,
    round(100.0 * sum(f.margem_contribuicao)
               / nullif(sum(f.receita_total), 0), 1)      AS margem_pct,
    round(avg(f.desconto_pct), 1)                         AS desconto_medio,
    round(sum(f.frete_subsidiado), 2)                     AS frete_absorvido
FROM fct_vendas f
JOIN dim_produto p ON p.sk_produto = f.sk_produto
WHERE f.custo_confiavel
GROUP BY ALL
HAVING sum(f.margem_contribuicao) < 0
   AND sum(f.receita_total) > 1000
ORDER BY margem_contribuicao;
""")

con.sql("SELECT * FROM an_produtos_prejuizo LIMIT 12").df()

### 09. Desempenho da equipe comercial

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW an_vendedor AS
SELECT
    v.nome_vendedor,
    round(100.0 * v.taxa_comissao, 1)                     AS taxa_comissao_pct,
    count(DISTINCT f.numero_pedido)                       AS pedidos,
    round(sum(f.receita_total), 2)                        AS receita,
    round(sum(f.receita_total) / count(DISTINCT f.numero_pedido), 2) AS ticket_medio,
    round(avg(f.desconto_pct), 1)                         AS desconto_medio,
    round(sum(f.custo_comissao), 2)                       AS comissao_paga,
    round(sum(f.margem_contribuicao), 2)                  AS margem,
    round(100.0 * sum(f.margem_contribuicao)
               / nullif(sum(f.receita_total), 0), 1)      AS margem_pct,
    rank() OVER (ORDER BY sum(f.receita_total) DESC)      AS posicao_receita,
    rank() OVER (ORDER BY sum(f.margem_contribuicao)
                        / nullif(sum(f.receita_total), 0) DESC) AS posicao_margem
FROM fct_vendas f
JOIN dim_vendedor v ON v.sk_vendedor = f.sk_vendedor
WHERE f.custo_confiavel
GROUP BY ALL
ORDER BY margem DESC;
""")

con.sql("SELECT * FROM an_vendedor").df()

## Exportação para o Looker Studio

Uma view única no grão do item, que permite ao painel agregar e filtrar por qualquer dimensão.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW painel AS
SELECT
    t.data,
    t.ano_mes,
    f.numero_pedido,
    c.razao_social                AS cliente,
    c.segmento,
    c.regiao,
    v.nome_vendedor               AS vendedor,
    p.descricao                   AS produto,
    p.categoria,
    f.tipo_operacao,
    f.quantidade,
    f.desconto_pct,
    f.receita_total,
    f.custo_produto,
    f.custo_entrega,
    f.custo_comissao,
    f.frete_subsidiado,
    f.margem_contribuicao
FROM fct_vendas f
JOIN dim_tempo    t ON t.sk_tempo    = f.sk_tempo
JOIN dim_produto  p ON p.sk_produto  = f.sk_produto
JOIN dim_vendedor v ON v.sk_vendedor = f.sk_vendedor
JOIN dim_cliente  c ON c.sk_cliente  = f.sk_cliente
WHERE f.custo_confiavel;
""")

con.sql("SELECT count(*) AS linhas, min(data) AS de, max(data) AS ate FROM painel").df()

In [ ]:
from google.colab import auth
from google.auth import default
import gspread

auth.authenticate_user()
credenciais, _ = default()
conexao = gspread.authorize(credenciais)

NOME = "Casa Verde - Painel SQL"
try:
    planilha = conexao.open(NOME)
except Exception:
    planilha = conexao.create(NOME)

abas = {
    "painel": "painel",
    "por_categoria": "an_categoria",
    "por_mes": "an_mensal",
    "por_desconto": "an_desconto",
    "clientes": "an_clientes_abc",
    "frete": "an_frete",
}

for nome, view in abas.items():
    df = con.sql(f"SELECT * FROM {view}").df()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d")
    df = df.where(pd.notna(df), "")

    try:
        aba = planilha.worksheet(nome)
        aba.clear()
    except Exception:
        aba = planilha.add_worksheet(title=nome, rows=len(df) + 10, cols=len(df.columns) + 2)

    aba.update(values=[list(df.columns)] + df.values.tolist(), range_name="A1")
    aba.freeze(rows=1)
    print(f"{nome:<14} {len(df):>6} linhas")

try:
    planilha.del_worksheet(planilha.worksheet("Sheet1"))
except Exception:
    pass

print()
print(planilha.url)

## Exportação local

Para quem quiser o banco pronto em vez da planilha.

In [ ]:
con.execute("EXPORT DATABASE 'saida' (FORMAT PARQUET)")
con.sql("SELECT * FROM painel").df().to_csv("painel.csv", index=False)
print("saida/ e painel.csv gravados")